# Lab 6: Spatial Autocorrelation in African Night-Lights — Moran's I

This notebook implements Lab 6 from *The New Regional Economics*.

**What you will learn:**
1. Build a spatial weight matrix from geographic adjacency
2. Compute global Moran's I to test for spatial autocorrelation in night-lights
3. Residualize on governance to test whether clustering persists after controls
4. Use permutation tests for inference (avoiding distributional assumptions)

**Prerequisites:** numpy, pandas, scipy (all pre-installed in Colab)

**Estimated time:** 2–3 hours

---

## 0. Setup

Clone the repository and install dependencies.

In [ ]:
# Clone the repository (only needed once per Colab session)
!git clone https://github.com/laurencehw/regional-economics.git 2>/dev/null || echo "Already cloned"
import os
os.chdir("regional-economics")

# Install dependencies
!pip install -q numpy pandas scipy

In [ ]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add lab code to path
sys.path.insert(0, "labs/lab6_africa/code")

from lab6_africa_moran_scaffold import (
    build_weight_matrix,
    row_standardize,
    prepare_cross_section,
    morans_i,
    permutation_p_value,
    residualize,
    synthetic_inputs,
)

print("Setup complete.")

## 1. Generate Synthetic Data

The synthetic dataset has 15 regions on a 3×5 grid with spatially
autocorrelated night-lights (by design). This lets us verify that
Moran's I detects the spatial pattern we built in.

In [ ]:
panel_df, adjacency_df = synthetic_inputs()

print(f"Panel: {panel_df.shape[0]} rows, {panel_df['region'].nunique()} regions")
print(f"Adjacency: {adjacency_df.shape[0]} edges")
display(panel_df.head(10))
display(adjacency_df.head(10))

## 2. Build the Spatial Weight Matrix

The weight matrix **W** captures geographic adjacency.
Unlike Lab 1 (trade-weighted), Lab 6 uses binary contiguity:
W(i,j) = 1 if regions i and j share a border, 0 otherwise.

In [ ]:
regions = sorted(panel_df["region"].unique())

w_raw = build_weight_matrix(adjacency_df, regions, "region", "neighbor", "weight")
w = row_standardize(w_raw)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(w_raw, cmap="Blues")
axes[0].set_title("Raw Adjacency Matrix")
axes[0].set_xticks(range(len(regions)))
axes[0].set_xticklabels(regions, rotation=90, fontsize=7)
axes[0].set_yticks(range(len(regions)))
axes[0].set_yticklabels(regions, fontsize=7)
plt.colorbar(im0, ax=axes[0])

im1 = axes[1].imshow(w, cmap="Blues")
axes[1].set_title("Row-Standardized W")
axes[1].set_xticks(range(len(regions)))
axes[1].set_xticklabels(regions, rotation=90, fontsize=7)
axes[1].set_yticks(range(len(regions)))
axes[1].set_yticklabels(regions, fontsize=7)
plt.colorbar(im1, ax=axes[1])

plt.tight_layout()
plt.show()

# Check symmetry
print(f"Symmetric: {np.allclose(w_raw, w_raw.T)}")
print(f"W density: {np.count_nonzero(w) / w.size:.2%}")

## 3. Compute Moran's I (Raw Night-Lights)

Global Moran's I measures the overall degree of spatial autocorrelation:

$$I = \frac{n}{S_0} \cdot \frac{\sum_i \sum_j w_{ij}(x_i - \bar{x})(x_j - \bar{x})}{\sum_i (x_i - \bar{x})^2}$$

- I > E[I] → positive spatial autocorrelation (clusters of similar values)
- I < E[I] → negative autocorrelation (checkerboard pattern)
- I ≈ E[I] → no spatial pattern

In [ ]:
y = panel_df["night_lights_mean"].to_numpy(dtype=float)

i_raw, i_expected = morans_i(y, w)
p_raw, perm_draws = permutation_p_value(y, w, permutations=999, seed=42)

print(f"Moran's I (raw):  {i_raw:.4f}")
print(f"Expected I (null): {i_expected:.4f}")
print(f"p-value (two-sided, 999 permutations): {p_raw:.4f}")
print(f"\nInterpretation: {'Significant' if p_raw < 0.05 else 'Not significant'} "
      f"spatial autocorrelation at 5% level")

In [ ]:
# Permutation distribution
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(perm_draws, bins=40, alpha=0.7, color="steelblue", edgecolor="white",
        label="Permutation distribution")
ax.axvline(i_raw, color="red", linewidth=2, label=f"Observed I = {i_raw:.4f}")
ax.axvline(i_expected, color="gray", linewidth=1, linestyle="--",
           label=f"Expected I = {i_expected:.4f}")
ax.set_xlabel("Moran's I")
ax.set_ylabel("Frequency")
ax.set_title("Permutation Test for Moran's I")
ax.legend()
plt.tight_layout()
plt.show()

## 4. Residualize on Governance

Is the spatial clustering just a reflection of governance quality being
spatially clustered? We regress night-lights on governance, then compute
Moran's I on the residuals.

In [ ]:
governance = panel_df["governance_score"].to_numpy(dtype=float).reshape(-1, 1)
resid = residualize(y, governance)

i_resid, i_exp_resid = morans_i(resid, w)
p_resid, _ = permutation_p_value(resid, w, permutations=999, seed=43)

print(f"Moran's I (raw):       {i_raw:.4f}  (p = {p_raw:.4f})")
print(f"Moran's I (residual):  {i_resid:.4f}  (p = {p_resid:.4f})")
print(f"\nReduction: {((i_raw - i_resid) / i_raw * 100):.1f}%")
print(f"\nGovernance {'explains some' if i_resid < i_raw else 'does not explain'} "
      f"of the spatial clustering.")
print(f"Residual clustering is {'still significant' if p_resid < 0.05 else 'not significant'} "
      f"— spatial patterns persist beyond governance.")

## 5. Exercises

### Exercise 1: K-Nearest Neighbors
Replace the contiguity-based W with a k-nearest-neighbors matrix (k=3, 5).
How sensitive is Moran's I to the choice of spatial weights?

### Exercise 2: Local Moran's I
Global Moran's I tells you *whether* there is clustering, not *where*.
Implement Local Moran's I (LISA) for each region and identify
High-High and Low-Low clusters.

### Exercise 3: Temporal Stability
If you have multi-year data, compute Moran's I for each year.
Is the spatial pattern stable or evolving?

### Exercise 4: Interpretation
Why might night-lights be spatially autocorrelated even after controlling
for governance? Propose at least two mechanisms (hint: think about
trade, infrastructure, and natural resources).

In [ ]:
# Space for student work
